In [ ]:
# Notebook: 02_EDA_and_Modeling.ipynb

# 1️⃣ Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import os

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, RocCurveDisplay

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from imblearn.over_sampling import SMOTE

# Ensure reports folder exists
if not os.path.exists("reports"):
    os.makedirs("reports")

# 2️⃣ Load dataset
cols = [
    "duration","protocol_type","service","flag","src_bytes","dst_bytes","land",
    "wrong_fragment","urgent","hot","num_failed_logins","logged_in","num_compromised",
    "root_shell","su_attempted","num_root","num_file_creations","num_shells",
    "num_access_files","num_outbound_cmds","is_host_login","is_guest_login",
    "count","srv_count","serror_rate","srv_serror_rate","rerror_rate","srv_rerror_rate",
    "same_srv_rate","diff_srv_rate","srv_diff_host_rate","dst_host_count",
    "dst_host_srv_count","dst_host_same_srv_rate","dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate","dst_host_srv_diff_host_rate","dst_host_serror_rate",
    "dst_host_srv_serror_rate","dst_host_rerror_rate","dst_host_srv_rerror_rate",
    "label","difficulty"
]

df = pd.read_csv("data/KDDTrain+.txt", names=cols)

# 3️⃣ Encode categorical features
categorical_cols = ["protocol_type", "service", "flag"]
le_dict = {}

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    le_dict[col] = le

# 4️⃣ Separate features and target
X = df.drop(columns=["label", "difficulty"])
y = df["label"]

# 5️⃣ Handle class imbalance safely with SMOTE
print("Original class counts:", Counter(y))
min_class_size = min(Counter(y).values())
k_neighbors = min(5, min_class_size - 1)

if k_neighbors < 1:
    print("Warning: Some classes have only 1 sample. SMOTE skipped.")
    X_res, y_res = X, y
else:
    smote = SMOTE(random_state=42, k_neighbors=k_neighbors)
    X_res, y_res = smote.fit_resample(X, y)
    print("Resampled class counts:", Counter(y_res))

# 5️⃣b Encode target labels numerically for XGBoost
target_le = LabelEncoder()
y_res_encoded = target_le.fit_transform(y_res)

# Save resampled dataset for reproducibility
resampled_df = pd.DataFrame(X_res, columns=X.columns)
resampled_df['label'] = y_res_encoded  # numeric
resampled_df.to_csv("data/KDDTrain_resampled.csv", index=False)

# 6️⃣ Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X_res, y_res_encoded, test_size=0.2, random_state=42, stratify=y_res_encoded
)

# 7️⃣ Scale features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# 8️⃣ Helper function for evaluation
def evaluate_model(model, X_test, y_test, model_name, label_encoder=None):
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average="weighted")
    rec = recall_score(y_test, y_pred, average="weighted")
    f1 = f1_score(y_test, y_pred, average="weighted")
    
    print(f"=== {model_name} ===")
    print(f"Accuracy: {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall: {rec:.4f}")
    print(f"F1-score: {f1:.4f}\n")
    
    # If numeric labels, convert back to original attack names
    if label_encoder is not None:
        y_test_labels = label_encoder.inverse_transform(y_test)
        y_pred_labels = label_encoder.inverse_transform(y_pred)
    else:
        y_test_labels = y_test
        y_pred_labels = y_pred
    
    # Confusion matrix
    cm = confusion_matrix(y_test_labels, y_pred_labels)
    plt.figure(figsize=(12,10))
    sns.heatmap(cm, annot=False, cmap='Blues')
    plt.title(f"{model_name} - Confusion Matrix")
    plt.ylabel("Actual")
    plt.xlabel("Predicted")
    plt.savefig(f"reports/{model_name}_confusion_matrix.png")
    plt.close()
    
    # ROC Curve (multi-class only works for 2 classes here)
    if len(np.unique(y_test)) == 2:
        RocCurveDisplay.from_estimator(model, X_test, y_test)
        plt.savefig(f"reports/{model_name}_ROC_curve.png")
        plt.close()
    
    # Feature importance for tree-based models
    if hasattr(model, "feature_importances_"):
        feat_imp = pd.Series(model.feature_importances_, index=df.drop(columns=["label","difficulty"]).columns)
        feat_imp.sort_values().plot(kind='barh', figsize=(10,8))
        plt.title(f"{model_name} - Feature Importance")
        plt.savefig(f"reports/{model_name}_feature_importance.png")
        plt.close()

# 9️⃣ Train models
# Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
evaluate_model(lr, X_test, y_test, "Logistic_Regression", label_encoder=target_le)

# Random Forest
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)
evaluate_model(rf, X_test, y_test, "Random_Forest", label_encoder=target_le)

# XGBoost
xgb = XGBClassifier(use_label_encoder=False, eval_metric="mlogloss", random_state=42)
xgb.fit(X_train, y_train)
evaluate_model(xgb, X_test, y_test, "XGBoost", label_encoder=target_le)


Original class counts: Counter({'normal': 67343, 'neptune': 41214, 'satan': 3633, 'ipsweep': 3599, 'portsweep': 2931, 'smurf': 2646, 'nmap': 1493, 'back': 956, 'teardrop': 892, 'warezclient': 890, 'pod': 201, 'guess_passwd': 53, 'buffer_overflow': 30, 'warezmaster': 20, 'land': 18, 'imap': 11, 'rootkit': 10, 'loadmodule': 9, 'ftp_write': 8, 'multihop': 7, 'phf': 4, 'perl': 3, 'spy': 2})
Resampled class counts: Counter({'normal': 67343, 'neptune': 67343, 'warezclient': 67343, 'ipsweep': 67343, 'portsweep': 67343, 'teardrop': 67343, 'nmap': 67343, 'satan': 67343, 'smurf': 67343, 'pod': 67343, 'back': 67343, 'guess_passwd': 67343, 'ftp_write': 67343, 'multihop': 67343, 'rootkit': 67343, 'buffer_overflow': 67343, 'imap': 67343, 'warezmaster': 67343, 'phf': 67343, 'land': 67343, 'loadmodule': 67343, 'spy': 67343, 'perl': 67343})
=== Logistic_Regression ===
Accuracy: 0.9888
Precision: 0.9887
Recall: 0.9888
F1-score: 0.9887

=== Random_Forest ===
Accuracy: 0.9999
Precision: 0.9999
Recall: 0.9

c:\dev\network_intrusion_detection\venv\Lib\site-packages\xgboost\training.py:199: UserWarning: [11:32:25] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


=== XGBoost ===
Accuracy: 0.9999
Precision: 0.9999
Recall: 0.9999
F1-score: 0.9999

